In [ ]:
import os
import json
import copy
import numpy as np

from cebra.data.helper import _require_numpy_array, OrthogonalProcrustesAlignment

### SET BASE_DIR ###
BASE_DIR = '/data'

In [ ]:
## Function to ensemble embeddings using Orthogonal Procrustes alignment on a subsample of the data

def ensemble_embeddings_subsampled(
    embeddings,
    subsample_fraction=0.02,
    top_k=5,
    ref_index=0,
    seed=0,
):
    """
    Align embeddings using Orthogonal Procrustes on a subsample,
    then average them.

    Args:
        embeddings: list of (n_samples, n_features) numpy arrays
        subsample_fraction: fraction of samples used to fit alignment
        top_k: passed to OrthogonalProcrustesAlignment
        ref_index: index of reference embedding
        seed: RNG seed for subsampling

    Returns:
        averaged_embedding: (n_samples, n_features)
    """
    embeddings = [_require_numpy_array(e) for e in embeddings]
    n_samples = embeddings[0].shape[0]

    # shape sanity check
    for e in embeddings[1:]:
        if e.shape != embeddings[0].shape:
            raise ValueError(f"Inconsistent shapes: {e.shape} vs {embeddings[0].shape}")

    subsample = max(1, int(subsample_fraction * n_samples))
    rng = np.random.default_rng(seed)
    subset_idx = rng.choice(n_samples, subsample, replace=False)

    aligner = OrthogonalProcrustesAlignment(top_k=top_k, subsample=subsample)

    ref_embedding = embeddings[ref_index]
    joint_embedding = copy.deepcopy(ref_embedding)

    for i, emb in enumerate(embeddings):
        if i == ref_index:
            continue

        aligner.fit(
            ref_data=ref_embedding[subset_idx],
            data=emb[subset_idx],
            ref_label=None,
            label=None,
        )
        joint_embedding += aligner.transform(emb)

    joint_embedding /= len(embeddings)
    return joint_embedding.astype(np.float32, copy=False)


In [ ]:
# =========================
# CONFIG 
# =========================
SAVE_ROOT = os.path.join(BASE_DIR, "CEBRA_project/Scripts_to_publish/xCEBRA_models_to_publish")
WEIGHT_TAG = "regularizer_weight_10p0"
SUBSAMPLE_FRACTION = 0.2
TOP_K = 5
REF_INDEX = 0

# =========================
# Discover embeddings
# =========================
weight_dir = os.path.join(SAVE_ROOT, WEIGHT_TAG)
assert os.path.isdir(weight_dir), f"Missing directory: {weight_dir}"

run_dirs = sorted(
    d for d in os.listdir(weight_dir)
    if d.startswith("run_") and os.path.isdir(os.path.join(weight_dir, d))
)

embeddings = []
used_runs = []

for r in run_dirs:
    emb_path = os.path.join(weight_dir, r, "embedding.npy")
    if os.path.exists(emb_path):
        embeddings.append(np.load(emb_path))
        used_runs.append(r)

assert len(embeddings) > 0, "No embeddings found for weight=10"

print(f"Loaded {len(embeddings)} runs | shape = {embeddings[0].shape}")

# =========================
# Output paths
# =========================
out_dir = os.path.join(SAVE_ROOT, "ensembled_embeddings", WEIGHT_TAG)
out_npz  = os.path.join(out_dir, "ensemble_avg_weight10p0.npz")
out_meta = os.path.join(out_dir, "ensemble_avg_weight10p0_meta.json")

# Skip if the ensembled embedding already exists
if os.path.exists(out_npz) and os.path.exists(out_meta):
    print("⏭  Ensemble already exists, skipping:")
    print(" ", out_npz)
else:
    # =========================
    # Ensemble (WITH FIXED SEED)
    # =========================
    avg_embedding = ensemble_embeddings_subsampled(
        embeddings=embeddings,
        subsample_fraction=SUBSAMPLE_FRACTION,
        top_k=TOP_K,
        ref_index=REF_INDEX,
    )

    # =========================
    # Save output
    # =========================
    os.makedirs(out_dir, exist_ok=True)

    np.savez_compressed(out_npz, combined_avg=avg_embedding)

    meta = {
        "weight": 10.0,
        "n_runs": len(embeddings),
        "runs_used": used_runs,
        "embedding_shape": list(avg_embedding.shape),
        "method": {
            "alignment": "OrthogonalProcrustesAlignment",
            "top_k": TOP_K,
            "subsample_fraction": SUBSAMPLE_FRACTION,
            "ref_index": REF_INDEX,
        }
    }

    with open(out_meta, "w") as f:
        json.dump(meta, f, indent=2)

    print("✅ Saved ensemble:")
    print(" ", out_npz)
    print(" ", out_meta)
